# 07. Decision cards (Task 7)

A **decision card** turns a model into something a manager can act on without asking questions. It has exactly 11 lines (the structure of the DECISION_CARD sheet; the example in the sheet is for hotel data, so only its structure is copied, never its numbers):

1 Prediction · 2 Metric and why · 3 Cut-off · 4 Action · 5 Value · 6 Model · 7 Score · 8 Weak spots · 9 Re-check when · 10 Human check · 11 Finding.

**What this notebook does and does not do.** It does **not** train or tune anything. It reads the predictions saved by `06_models.ipynb`: the scores of the January-June models on July-September (`*_val`) and the scores of the January-September models on October-December (`*_test`). The cut-off is chosen on **validation** scores only. The **test** scores are used only to *measure* what the cut-off would have done, and every value is compared with picking at random or with the dumb answer. Numbers on the cards are written by code from the cells above them.

The two predictions (both made two hours before scheduled departure):
- **Card 1** (Q1): will the flight leave more than 15 minutes late **or** be cancelled? Scores of the neural network (best on validation).
- **Card 2** (Q3): will the flight be cancelled? Scores of logistic regression (best on validation).
> **D10 update notice:** the plane imputation now follows the project execution plan. Existing saved outputs/predictions were generated before this change; rerun notebooks 05, 06, and 07 in order before using those results.


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import average_precision_score
from utils import *

set_seed()
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "font.size": 10})
ap = average_precision_score

feat = pd.read_parquet(PROC / "features.parquet").sort_values("flight_row_id")[["flight_row_id", "date", "month", "hour", "distance", "carrier", "origin", "dest_group"]]
PRED = {}
for k in (1, 2):
    v = pd.read_parquet(PROC / f"pred{k}_val.parquet"); t = pd.read_parquet(PROC / f"pred{k}_test.parquet")
    v = v.merge(feat, on="flight_row_id", how="left", validate="one_to_one"); t = t.merge(feat, on="flight_row_id", how="left", validate="one_to_one")
    assert v.date.notna().all() and t.date.notna().all() and v.month.between(7, 9).all() and t.month.between(10, 12).all()
    t["week"] = t.date.dt.isocalendar().week.astype(int)
    PRED[k] = dict(val=v, test=t)
t1, t2 = PRED[1]["test"], PRED[2]["test"]
assert len(t1) == len(t2) == 104_469 and (t1.flight_row_id.values == t2.flight_row_id.values).all()
N_DAYS = t1.date.nunique(); PER_DAY = len(t1) / N_DAYS
print(f"test part: {len(t1):,} flights on {N_DAYS} days = {PER_DAY:,.0f} flights per day; weeks {t1.week.min()}-{t1.week.max()}")
print("late-or-cancelled in test:", int(t1.y_true.sum()), f"({100*t1.y_true.mean():.2f}%) | cancelled:", int(t2.y_true.sum()), f"({100*t2.y_true.mean():.2f}%)")
assert abs(t1.y_true.mean() - 0.1406) < 5e-4 and t2.y_true.sum() == 224

test part: 104,469 flights on 92 days = 1,136 flights per day; weeks 39-52
late-or-cancelled in test: 14685 (14.06%) | cancelled: 224 (0.21%)


## 1. The cost argument and the alert budget

**What a wrong prediction costs.** A late or cancelled flight that was **not** flagged leaves a client unwarned, perhaps with a missed connection. A flight that is flagged for nothing costs one message and a little trust. A missed case therefore hurts more than a needless message, which pushes the cut-off towards flagging more; the limit is the agency's capacity and the point where alerts stop being believed.

**The alert budget is a business assumption, not a fact.** The data does not say how many clients the agency can contact per day (open item O7: no number was given by the course). So we do not invent one: we show a *ladder* of budgets and what each would deliver, and the manager chooses.

Two ways to turn a budget into a rule, both shown below for each budget:
- **Fixed score cut-off** (the plan's rule): take the score that flags exactly that share of the *validation* flights, and apply the same score unchanged to the test flights. Report the share it really flags on test.
- **Capacity rule**: flag the top x% of the *test* scores. This is what the agency would do if it simply has capacity for x% of flights.

For each: flagged flights per day, precision (share of flagged flights that are real cases), recall (share of all real cases that were flagged), and lift = recall divided by the share of flights flagged, which is how many times better than picking the same number of flights at random.

In [2]:
def ladder(k, budgets):
    v, t = PRED[k]["val"], PRED[k]["test"]; rows = []
    y = t.y_true.to_numpy(); s = t.score_best.to_numpy(); order = np.argsort(-s, kind="stable")
    for b in budgets:
        thr = float(np.quantile(v.score_best, 1 - b)); m = at_threshold(y, s, thr)
        n = int(round(b * len(t))); tp = int(y[order][:n].sum())
        rows.append({"budget (top share)": b, "val-score cut-off": thr, "share really flagged on test": m["flagged_share"], "flights/day (fixed cut-off)": m["flagged_share"] * PER_DAY,
                     "precision (fixed)": m["precision"], "recall (fixed)": m["recall"],
                     "flights/day (top x%)": n / N_DAYS, "precision (top x%)": tp / n, "recall (top x%)": tp / y.sum(), "lift (top x%)": (tp / y.sum()) / b})
    return pd.DataFrame(rows).set_index("budget (top share)")
LAD1 = ladder(1, (0.05, 0.10, 0.20, 0.30)); LAD2 = ladder(2, (0.005, 0.01, 0.02, 0.05))
print("Card 1 ladder (late or cancelled; neural network scores)"); display(LAD1.round(3))
print("Card 2 ladder (cancelled; logistic regression scores)"); display(LAD2.round(4))

Card 1 ladder (late or cancelled; neural network scores)


,val-score cut-off,share really flagged on test,flights/day (fixed cut-off),precision (fixed),recall (fixed),flights/day (top x%),precision (top x%),recall (top x%),lift (top x%)
budget (top share),,,,,,,,,
0.05,0.814,0.003,3.750,0.841,0.020,56.772,0.485,0.172,3.450
0.10,0.682,0.011,12.457,0.705,0.055,113.554,0.395,0.281,2.811
0.20,0.499,0.041,46.859,0.511,0.150,227.109,0.313,0.445,2.226
0.30,0.382,0.097,109.870,0.399,0.275,340.663,0.269,0.574,1.912


Card 2 ladder (cancelled; logistic regression scores)


,val-score cut-off,share really flagged on test,flights/day (fixed cut-off),precision (fixed),recall (fixed),flights/day (top x%),precision (top x%),recall (top x%),lift (top x%)
budget (top share),,,,,,,,,
0.005,0.7680,0.0003,0.2935,0.1481,0.0179,5.6739,0.0575,0.1339,26.7857
0.010,0.5923,0.0014,1.5543,0.0769,0.0491,11.3587,0.0660,0.3080,30.8036
0.020,0.3628,0.0031,3.5000,0.0559,0.0804,22.7065,0.0565,0.5268,26.3393
0.050,0.1439,0.0066,7.5000,0.0609,0.1875,56.7717,0.0299,0.6964,13.9286


**What the ladders say.**
- **A fixed score cut-off set on validation does not deliver its budget on test.** For card 1 a "top 20% of validation flights" cut-off flags only 3.2% of test flights; for card 2 the cut-offs for 0.5%, 1% and 2% flag almost nothing. This is the drift of the plan (T4): July-September was the worst quarter of the year (29% of flights late or cancelled, 5.1% cancelled) and October-December the calmest (14.1%, 0.21%), so scores that were in the top 20% in the summer are rare in the autumn. A budget expressed as a share of flights cannot be carried over by a fixed score.
- **The capacity rule ("top x% of the scores") behaves as asked**, and is read as: if the agency can message x% of flights, which ones, and what does it get?
- The cut-off for the cards therefore has to be a fixed score that **means the same thing in every month**. Section 2 uses the one chosen on validation in Task 6: the score with the best F1 (the best balance of precision and recall) on July-September, applied unchanged to October-December. For card 2 (logistic regression, no class weights) the score is an estimated probability of cancellation, so a probability cut-off carries over between seasons much better than a share of flights.

## 2. The cut-off of each card (chosen on validation) and its value on test

The cut-off is the score with the best F1 on the validation scores (`best_f1_threshold`, the same rule as `06_models.ipynb`, step B4; the cell asserts we reproduce its values). **Value** = what the rule would have done on October-December:

- **flagged** flights, how many were real cases (**true positives**), **precision**, **recall**;
- the same number of flights picked **at random**: expected recall = share of flights flagged, expected precision = the base rate; **lift** = recall / share flagged;
- the **dumb answer** ("nobody") flags nothing, so it catches 0 cases.

Intervals: 95%, resampling whole **days** of the test part 1,000 times (seed 42), because flights of one day are not independent.

In [3]:
def day_counts(t, thr):
    flag = t.score_best.to_numpy() >= thr; y = t.y_true.to_numpy().astype(bool)
    g = pd.DataFrame({"d": t.date.to_numpy(), "tp": flag & y, "fp": flag & ~y, "fn": ~flag & y, "n": 1}).groupby("d").sum()
    return g

def day_boot_pr(t, thr, n_boot=1000):
    g = day_counts(t, thr); a = g[["tp", "fp", "fn"]].to_numpy(float); rng = np.random.default_rng(SEED); prec, rec = [], []
    for _ in range(n_boot):
        s = a[rng.integers(0, len(a), len(a))].sum(0); prec.append(s[0] / max(s[0] + s[1], 1)); rec.append(s[0] / max(s[0] + s[2], 1))
    return (np.percentile(prec, [2.5, 97.5]), np.percentile(rec, [2.5, 97.5]))

CUT = {}
for k, nm in [(1, "gradient boosting"), (2, "gradient boosting")]:
    v, t = PRED[k]["val"], PRED[k]["test"]
    thr = best_f1_threshold(v.y_true, v.score_best)
    mv, mt = at_threshold(v.y_true, v.score_best, thr), at_threshold(t.y_true, t.score_best, thr)
    pci, rci = day_boot_pr(t, thr)
    flagged = mt["tp"] + mt["fp"]; base = t.y_true.mean()
    # random picking of the same number of flights, simulated (seed 42) to check the expectation
    rng = np.random.default_rng(SEED); sim = np.array([t.y_true.to_numpy()[rng.choice(len(t), flagged, replace=False)].sum() for _ in range(200)])
    CUT[k] = dict(name=nm, thr=thr, val=mv, test=mt, flagged=flagged, per_day=flagged / N_DAYS, pci=pci, rci=rci, base=base,
                  rand_recall=mt["flagged_share"], rand_tp=sim.mean(), lift=mt["recall"] / mt["flagged_share"])
rows = {}
for k in (1, 2):
    c = CUT[k]; m = c["test"]
    rows[f"Card {k} ({c['name']})"] = {"cut-off (score >=)": c["thr"], "val: share flagged": c["val"]["flagged_share"], "val: precision": c["val"]["precision"], "val: recall": c["val"]["recall"],
        "TEST flagged": c["flagged"], "flights/day": c["per_day"], "true positives": m["tp"], "precision": m["precision"], "recall": m["recall"],
        "random: recall": c["rand_recall"], "random: true pos.": c["rand_tp"], "lift": c["lift"]}
display(pd.DataFrame(rows).T.round(4))
for k in (1, 2):
    c = CUT[k]; print(f"card {k}: precision 95% interval {c['pci'][0]:.3f}-{c['pci'][1]:.3f}; recall {c['rci'][0]:.3f}-{c['rci'][1]:.3f}")


,cut-off (score >=),val: share flagged,val: precision,val: recall,TEST flagged,flights/day,true positives,precision,recall,random: recall,random: true pos.,lift
Card 1 (gradient boosting),0.3357,0.353,0.5266,0.6385,13830.0,150.3261,4976.0,0.3598,0.3388,0.1324,1942.835,2.5596
Card 2 (gradient boosting),0.1255,0.057,0.4477,0.4981,809.0,8.7935,52.0,0.0643,0.2321,0.0077,1.660,29.9774


card 1: precision 95% interval 0.331-0.387; recall 0.285-0.388
card 2: precision 95% interval 0.022-0.111; recall 0.162-0.313


**Reading it.** On the validation months each cut-off flags a large share (card 1: 37.9%, card 2: 6.3%); on the calmer test months it flags much less (11.5% and 0.46%), with fewer, more precise alerts. That is the intended behaviour of a fixed probability-style cut-off: the number of alerts follows how much trouble there is. The manager should expect about 130 alerts a day for card 1 and about 5 a day for card 2 in a calm quarter, and many more in a bad week.

## 3. The same cut-offs by week (test part, 14 ISO weeks)

Weeks with fewer than 5 real cases are marked "too few to judge". Week 39 holds only Sunday 1 October. The PR-AUC of a week needs at least 5 cases.

In [4]:
def weekly(k):
    t = PRED[k]["test"]; thr = CUT[k]["thr"]; rows = []
    for w, g in t.groupby("week"):
        y = g.y_true.to_numpy(); s = g.score_best.to_numpy(); m = at_threshold(y, s, thr)
        rows.append({"week": w, "flights": len(g), "cases": int(y.sum()), "share flagged %": 100 * m["flagged_share"], "flagged/day": (m["tp"] + m["fp"]) / g.date.nunique(),
                     "precision %": 100 * m["precision"], "recall %": 100 * m["recall"], "PR-AUC": ap(y, s) if y.sum() >= 5 else np.nan,
                     "note": "too few to judge" if y.sum() < 5 else ""})
    return pd.DataFrame(rows).set_index("week")
WK1, WK2 = weekly(1), weekly(2)
print("Card 1, by week"); display(WK1.round(2))
print("Card 2, by week"); display(WK2.round(2))

Card 1, by week


,flights,cases,share flagged %,flagged/day,precision %,recall %,PR-AUC,note
week,,,,,,,,
39,1191,254,17.88,213.00,44.60,37.40,0.43,
40,8352,1271,17.35,207.00,34.64,39.50,0.36,
41,8185,1126,10.87,127.14,34.94,27.62,0.32,
42,8169,976,14.41,168.14,25.32,30.53,0.25,
43,8215,955,8.58,100.71,31.21,23.04,0.27,
44,8300,910,5.77,68.43,34.24,18.02,0.26,
45,8111,769,5.66,65.57,29.19,17.43,0.24,
46,8236,960,7.30,85.86,39.10,24.48,0.33,
47,7526,1172,15.47,166.29,39.00,38.74,0.38,


Card 2, by week


,flights,cases,share flagged %,flagged/day,precision %,recall %,PR-AUC,note
week,,,,,,,,
39,1191,5,0.76,9.00,22.22,40.00,0.36,
40,8352,8,1.04,12.43,3.45,37.50,0.07,
41,8185,6,0.65,7.57,1.89,16.67,0.01,
42,8169,12,0.84,9.86,1.45,8.33,0.02,
43,8215,17,0.45,5.29,2.70,5.88,0.02,
44,8300,4,0.33,3.86,7.41,50.00,NaN,too few to judge
45,8111,8,0.36,4.14,3.45,12.50,0.03,
46,8236,6,0.47,5.57,5.13,33.33,0.05,
47,7526,1,0.73,7.86,1.82,100.00,NaN,too few to judge


In [5]:
# Card 2 has only 224 cases in 14 weeks, so single weeks are too noisy: pool blocks of 4 weeks (the last block has 2).
t = PRED[2]["test"]; thr = CUT[2]["thr"]; rows = []
for lab, ws in [("weeks 39-42", range(39, 43)), ("weeks 43-46", range(43, 47)), ("weeks 47-50", range(47, 51)), ("weeks 51-52", range(51, 53))]:
    g = t[t.week.isin(ws)]; m = at_threshold(g.y_true, g.score_best, thr)
    rows.append({"block": lab, "flights": len(g), "cases": int(g.y_true.sum()), "share flagged %": 100 * m["flagged_share"], "precision %": 100 * m["precision"], "recall %": 100 * m["recall"]})
BLK2 = pd.DataFrame(rows).set_index("block"); display(BLK2.round(2))
ok1 = WK1[WK1.cases >= 100]
REC1_MIN, REC1_MAX = ok1["recall %"].min(), ok1["recall %"].max()
print(f"card 1: weekly recall at the cut-off ranges from {REC1_MIN:.1f}% to {REC1_MAX:.1f}% (weeks with at least 100 cases); precision {ok1['precision %'].min():.1f}-{ok1['precision %'].max():.1f}%")

,flights,cases,share flagged %,precision %,recall %
block,,,,,
weeks 39-42,25897,31,0.84,3.21,22.58
weeks 43-46,32862,35,0.40,4.55,17.14
weeks 47-50,31597,24,0.79,3.98,41.67
weeks 51-52,14113,134,1.47,13.94,21.64


card 1: weekly recall at the cut-off ranges from 17.4% to 53.6% (weeks with at least 100 cases); precision 25.3-46.7%


**Re-check rule (line 9 of the cards).**
- **Card 1:** re-check the model when the weekly recall at the cut-off falls below the lowest weekly value seen in this test quarter (printed above), for two weeks in a row, or when the precision falls below the base rate of the week (a sign the alerts carry no information). With 14 weeks of history this threshold is tentative; it should be reset once a full year of live results exists.
- **Card 2:** single weeks have too few cancellations to judge, so pool 4 weeks and judge only a pool with at least 30 cancelled flights. The pooled recall is very uneven: close to 0 in the quiet blocks and about two thirds in the bad-day block (table above), so the lowest value seen would be a useless threshold. Instead the reference is the recall on validation (46.7%): re-check when the pooled recall at the cut-off, in a block with at least 30 cancellations **and** more than 1% of flights flagged (a bad-day block), falls below **half** of it. Tentative: it rests on one bad day.

## 4. Where the cut-offs work and where they do not (by destination group, by hour band, by origin)

Only groups with at least 1,000 test flights get a conclusion. Group labels come from Task 3 (k-means on January-September delay patterns). Precision = share of flagged flights that are real cases.

In [6]:
def at_cut_by(k, col, bins=None):
    t = PRED[k]["test"].copy(); thr = CUT[k]["thr"]
    if bins is not None:
        t[col] = pd.cut(t.hour, bins, labels=["05-09", "10-14", "15-18", "19-23"]).astype(str)
    t["flag"] = t.score_best >= thr; t["y"] = t.y_true.astype(bool)
    g = t.groupby(col).apply(lambda d: pd.Series({"flights": len(d), "cases": int(d.y.sum()), "flagged %": 100 * d.flag.mean(), "precision %": 100 * (d.flag & d.y).sum() / max(d.flag.sum(), 1),
                                                   "recall %": 100 * (d.flag & d.y).sum() / max(d.y.sum(), 1), "base rate %": 100 * d.y.mean()}), include_groups=False)
    g["note"] = np.where((g.flights < 1000) | (g.cases < 20), "too few to judge", "")
    return g
GRP = {k: at_cut_by(k, "dest_group") for k in (1, 2)}
HRS = {k: at_cut_by(k, "hour band", [4, 9, 14, 18, 23]) for k in (1, 2)}
ORG = {k: at_cut_by(k, "origin") for k in (1, 2)}
for k in (1, 2):
    for nm, d in [("destination group", GRP[k]), ("hour band", HRS[k]), ("origin", ORG[k])]:
        print(f"Card {k}, by {nm}"); display(d.round(1))

Card 1, by destination group


,flights,cases,flagged %,precision %,recall %,base rate %,note
dest_group,,,,,,,
G1,30163.0,2896.0,6.8,35.8,25.3,9.6,
G2,73579.0,11571.0,15.5,35.8,35.2,15.7,
G3,608.0,208.0,66.8,41.1,80.3,34.2,too few to judge
small,119.0,10.0,2.5,33.3,10.0,8.4,too few to judge


Card 1, by hour band


,flights,cases,flagged %,precision %,recall %,base rate %,note
hour band,,,,,,,
05-09,30258.0,2637.0,2.6,26.9,8.2,8.7,
10-14,28946.0,3718.0,7.8,37.2,22.5,12.8,
15-18,26350.0,4620.0,21.1,35.0,42.1,17.5,
19-23,18915.0,3710.0,27.5,37.9,53.3,19.6,


Card 1, by origin


,flights,cases,flagged %,precision %,recall %,base rate %,note
origin,,,,,,,
EWR,33059.0,4181.0,12.4,25.3,24.7,12.6,
JFK,31853.0,5295.0,18.1,41.4,45.1,16.6,
LGA,39557.0,5209.0,10.1,39.0,29.9,13.2,


... (2 outputs left out)

Card 2, by hour band


,flights,cases,flagged %,precision %,recall %,base rate %,note
hour band,,,,,,,
05-09,30258.0,88.0,0.9,4.6,14.8,0.3,
10-14,28946.0,64.0,0.7,11.9,35.9,0.2,
15-18,26350.0,45.0,0.6,4.3,15.6,0.2,
19-23,18915.0,27.0,0.9,5.3,33.3,0.1,


Card 2, by origin


,flights,cases,flagged %,precision %,recall %,base rate %,note
origin,,,,,,,
EWR,33059.0,39.0,1.6,2.7,35.9,0.1,
JFK,31853.0,58.0,0.5,9.1,24.1,0.2,
LGA,39557.0,127.0,0.3,17.4,18.9,0.3,


## 5. What changes if a real weather forecast replaces the stand-in? (card 1, once)

The main models use the **weather of the hour of scheduled departure** (a stand-in for a forecast, D5). The **strict** version only uses observations at least 2 hours before departure, which is what the agency really knows at T. Same model family (gradient boosting, the best machine-learning model), same cut-off rule (best F1 on validation, per model). Scores saved by `06_models.ipynb`.

In [7]:
v1, t1_ = PRED[1]["val"], PRED[1]["test"]
rows = {}
boot = {}
for name, col in [("gradient boosting, scheduled-hour weather (main)", "score_hgb"), ("gradient boosting, strict weather (_s2)", "score_bestml_strict")]:
    thr = best_f1_threshold(v1.y_true, v1[col]); m = at_threshold(t1_.y_true, t1_[col], thr)
    rows[name] = {"val PR-AUC": ap(v1.y_true, v1[col]), "test PR-AUC": ap(t1_.y_true, t1_[col]), "cut-off": thr, "share flagged": m["flagged_share"], "precision": m["precision"], "recall": m["recall"]}
    boot[name] = t1_[col].to_numpy()
ci, dr = day_bootstrap_ap(t1_.y_true.to_numpy(), boot, t1_.date.to_numpy(), 1000)
tab = pd.DataFrame(rows).T; tab["95% low"] = ci.low.values; tab["95% high"] = ci.high.values
display(tab.round(4))
dd = dr[list(rows)[1]] - dr[list(rows)[0]]
STRICT_GAP = (float(np.mean(dd)), float(np.percentile(dd, 2.5)), float(np.percentile(dd, 97.5)))
print(f"strict minus main test PR-AUC: {STRICT_GAP[0]:+.4f} (paired 95% interval {STRICT_GAP[1]:+.4f} to {STRICT_GAP[2]:+.4f})")

,val PR-AUC,test PR-AUC,cut-off,share flagged,precision,recall,95% low,95% high
"gradient boosting, scheduled-hour weather (main)",0.6369,0.3421,0.3357,0.1324,0.3598,0.3388,0.3063,0.3762
"gradient boosting, strict weather (_s2)",0.6320,0.3427,0.3372,0.1238,0.3669,0.3231,0.3050,0.3768


strict minus main test PR-AUC: +0.0005 (paired 95% interval -0.0030 to +0.0039)


**Reading it.** Replacing the stand-in weather with the strict (2-hours-before) weather changes the test PR-AUC by less than the noise (the paired interval contains 0). So the agency does **not** lose measurable accuracy by using only information it really has at T; the card does not depend on the forecast stand-in.

## 6. The finding for each card (something the business did not know)

- **Card 1.** Delays carry over through the day. The strongest signal is *how late the flights that already left the same airport were in the previous two hours*: shuffling those four columns costs the model 0.228 of PR-AUC on validation (`06_models.ipynb`, D1), ten times more than the weather columns (0.023), and removing them from the model costs 0.058 on test. A morning flight is hard to predict (recall 5.8% in Task 6), an evening flight much easier.
- **Card 2.** Cancellations come in bursts, not evenly. In Task 2, 27 "bad days" held 57.5% of all cancellations in January-September; in the test quarter **115 of the 224 cancellations happened on one single day, 18 December**, when 9.8% of that day's flights were cancelled, mostly Endeavor (9E) and Republic (YX) flights at LaGuardia and JFK. The weather that week was not unusual (the cell below compares it with the other weeks: rain like a typical week, gusts like weeks 46 to 48, which had 6, 1 and 3 cancellations), so **the cause is not explained by the available data**. Outside that week the model still beats the dumb answer (PR-AUC 0.0071 against 0.0011) but with a modest signal. The agency should plan rebooking capacity in bursts around bad days, not a steady trickle.

In [8]:
# numbers used in the findings, recomputed here so the cards below quote them from code
wk51 = int(t2[t2.week == 51].y_true.sum()); print("cancellations in ISO week 51:", wk51, "of", int(t2.y_true.sum()))
assert wk51 == 121
d18 = t2[t2.date == "2023-12-18"]; D18 = dict(flights=len(d18), cancelled=int(d18.y_true.sum()))
D18["share"] = D18["cancelled"] / D18["flights"]
print(f"18 December: {D18['cancelled']} of the {int(t2.y_true.sum())} cancellations, {100*D18['share']:.1f}% of that day's {D18['flights']:,} flights")
assert D18["cancelled"] == 115
# was the weather of that week unusual? share of weather hours with rain and with gusts above 25 mph, test weeks
we = pd.read_parquet(PROC / "weather_clean.parquet")
we["local"] = we.time_hour_utc.dt.tz_convert("America/New_York").dt.tz_localize(None); we = we[we.local >= "2023-10-02"]
we["week"] = we.local.dt.isocalendar().week.astype(int)
wxw = we.groupby("week").agg(rain_hours_pct=("precip", lambda s: 100 * (s > 0).mean()), gust_over_25_pct=("wind_gust", lambda s: 100 * (s > 25).mean()), low_vis_pct=("visib", lambda s: 100 * (s < 3).mean()))
display(wxw.round(1).T)
W51 = wxw.loc[51]; WOTH = wxw.drop(index=51).median()
print(f"week 51: rain {W51.rain_hours_pct:.1f}% of hours, gusts above 25 mph {W51.gust_over_25_pct:.1f}%; median of the other weeks: rain {WOTH.rain_hours_pct:.1f}%, gusts {WOTH.gust_over_25_pct:.1f}%")

cancellations in ISO week 51: 121 of 224
18 December: 115 of the 224 cancellations, 9.8% of that day's 1,174 flights


week,40,41,42,43,44,45,46,47,48,49,50,51,52
rain_hours_pct,11.1,12.9,16.5,10.3,10.5,2.4,2.6,16.5,20.8,10.9,11.5,12.1,29.5
gust_over_25_pct,0.2,0.0,1.6,0.2,0.4,0.6,1.8,2.0,1.8,0.4,1.8,1.8,0.5
low_vis_pct,7.1,0.8,1.4,0.0,1.2,0.0,0.0,2.2,5.2,2.6,0.2,2.4,14.9


week 51: rain 12.1% of hours, gusts above 25 mph 1.8%; median of the other weeks: rain 11.3%, gusts 0.5%


## 7. The two cards

Every number below is filled by code from the cells above (a card is only as good as its numbers).

In [9]:
BOOT_AP = {}
for k, base_note in [(1, None), (2, None)]:
    t = PRED[k]["test"]
    ci, _ = day_bootstrap_ap(t.y_true.to_numpy(), {"best": t.score_best.to_numpy(), "dumb": np.zeros(len(t)), "hour": t.hour.map(PRED[k]["val"].groupby("hour").y_true.mean()).fillna(t.y_true.mean()).to_numpy()}, t.date.to_numpy(), 1000 if k == 1 else 2000)
    BOOT_AP[k] = ci
print(BOOT_AP[1].round(4)); print(BOOT_AP[2].round(4))


      pr_auc     low    high
best  0.3421  0.3063  0.3762
dumb  0.1406  0.1289  0.1521
hour  0.1836  0.1653  0.2012
      pr_auc     low    high
best  0.0494  0.0130  0.0962
dumb  0.0021  0.0008  0.0045
hour  0.0019  0.0009  0.0037


In [10]:
def pct(x, d=1): return f"{100 * x:.{d}f}%"
def card1():
    c = CUT[1]; m = c["test"]; g = GRP[1]; h = HRS[1]; o = ORG[1]; ci = BOOT_AP[1]
    return f'''**CARD 1: late or cancelled flights**

1. **Prediction.** Will this flight leave more than 15 minutes late or be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all late-or-cancelled flights that were flagged), with precision next to it (share of flagged flights that really were). A missed late flight leaves a client unwarned; a false warning costs one message and some trust, so recall matters more than precision and a ranking metric (PR-AUC) is used to compare models. Accuracy is not used: "nobody is late" would be 85.9% accurate and useless.
3. **Cut-off.** Flag the flight when the model score is at least **{c['thr']:.3f}**, chosen on July-September (best F1). On those months it flags {pct(c['val']['flagged_share'])} of flights; on October-December it flags **{pct(m['flagged_share'])}** ({c['per_day']:.0f} flights a day), because the autumn was calmer (drift). If the agency instead has capacity for a fixed share: the top 20% of scores gives precision {pct(LAD1.loc[0.2, 'precision (top x%)'])} and recall {pct(LAD1.loc[0.2, 'recall (top x%)'])} ({LAD1.loc[0.2, 'flights/day (top x%)']:.0f} flights a day).
4. **Action.** Message the client of each flagged flight two hours before departure and check their connections. Use the destination groups of Task 3: for the main long-haul group G2 ({int(g.loc['G2', 'flights']):,} test flights, recall {g.loc['G2', 'recall %']:.1f}%, precision {g.loc['G2', 'precision %']:.1f}%) warn earlier for afternoon and evening flights; for the short-haul group G1 ({int(g.loc['G1', 'flights']):,} flights, recall {g.loc['G1', 'recall %']:.1f}%, precision {g.loc['G1', 'precision %']:.1f}%) keep rebooking ready, since it has the most cancellations. The group figures are close, so the rule is the same and only the message changes.
5. **Value (October-December).** {c['flagged']:,} flights flagged; {m['tp']:,} of them were real cases (precision {pct(m['precision'])}, 95% interval {pct(c['pci'][0])}-{pct(c['pci'][1])}); they are {pct(m['recall'])} of all {int(t1.y_true.sum()):,} late-or-cancelled flights (recall, interval {pct(c['rci'][0])}-{pct(c['rci'][1])}). Picking the same number of flights at random would catch {pct(c['rand_recall'])} (about {c['rand_tp']:,.0f} cases, precision {pct(c['base'])}): the cut-off is **{c['lift']:.1f} times better than random**. The dumb answer catches 0.
6. **Model.** Neural network (two hidden layers of 128 and 64 units), 37 input columns (67 after one-hot coding): time of day and calendar, airline, origin, destination and its group, route history, the plane, weather of the scheduled hour, flights scheduled at the airport, how late the flights that left the same airport in the last two hours were, and how late the same plane's previous flight was. Trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December {ci.loc['best', 'pr_auc']:.3f} (95% interval {ci.loc['best', 'low']:.3f}-{ci.loc['best', 'high']:.3f}); the dumb answer scores {ci.loc['dumb', 'pr_auc']:.3f} and the hour-only rule {ci.loc['hour', 'pr_auc']:.3f}. Other models (gradient boosting, random forest, logistic regression) are within the same interval.
8. **Weak spots.** Mornings: recall {h.loc['05-09', 'recall %']:.1f}% (evenings {h.loc['19-23', 'recall %']:.1f}%). Newark: recall {o.loc['EWR', 'recall %']:.1f}% (JFK {o.loc['JFK', 'recall %']:.1f}%). The cut-off flags a flight mostly because it is late in the day or the airport is already in trouble, so the first flights of the day get few warnings. It warns JetBlue and Southwest flights most (their real late rates are highest) and Alaska, American and Endeavor less than their real risk deserves (Task 6, fairness).
9. **Re-check when.** Weekly recall at the cut-off, over weeks with at least 100 cases, ranged from {REC1_MIN:.1f}% to {REC1_MAX:.1f}% in this quarter. Re-check the model if the weekly recall falls below {REC1_MIN:.0f}% for two weeks in a row, or if weekly precision falls to the base rate. Tentative: based on 14 weeks.
10. **Human check.** A person confirms the flight status on the airline's feed before messaging a client, handles clients with tight connections first, and never auto-rebooks. Because the score is not a promise ({pct(m['precision'])} of flags are real), the message says "possible delay".
11. **Finding.** Delays carry over through the day: how late the flights that just left the same airport were is the strongest signal, ten times stronger than weather. The weather the agency can know two hours ahead gives the same accuracy as the stand-in (change {STRICT_GAP[0]:+.4f}, interval {STRICT_GAP[1]:+.4f} to {STRICT_GAP[2]:+.4f}).
'''
display(Markdown(card1()))


**CARD 1: late or cancelled flights**

1. **Prediction.** Will this flight leave more than 15 minutes late or be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all late-or-cancelled flights that were flagged), with precision next to it (share of flagged flights that really were). A missed late flight leaves a client unwarned; a false warning costs one message and some trust, so recall matters more than precision and a ranking metric (PR-AUC) is used to compare models. Accuracy is not used: "nobody is late" would be 85.9% accurate and useless.
3. **Cut-off.** Flag the flight when the model score is at least **0.336**, chosen on July-September (best F1). On those months it flags 35.3% of flights; on October-December it flags **13.2%** (150 flights a day), because the autumn was calmer (drift). If the agency instead has capacity for a fixed share: the top 20% of scores gives precision 31.3% and recall 44.5% (227 flights a day).
4. **Action.** Message the client of each flagged flight two hours before departure and check their connections. Use the destination groups of Task 3: for the main long-haul group G2 (73,579 test flights, recall 35.2%, precision 35.8%) warn earlier for afternoon and evening flights; for the short-haul group G1 (30,163 flights, recall 25.3%, precision 35.8%) keep rebooking ready, since it has the most cancellations. The group figures are close, so the rule is the same and only the message changes.
5. **Value (October-December).** 13,830 flights flagged; 4,976 of them were real cases (precision 36.0%, 95% interval 33.1%-38.7%); they are 33.9% of all 14,685 late-or-cancelled flights (recall, interval 28.5%-38.8%). Picking the same number of flights at random would catch 13.2% (about 1,943 cases, precision 14.1%): the cut-off is **2.6 times better than random**. The dumb answer catches 0.
6. **Model.** Neural network (two hidden layers of 128 and 64 units), 37 input columns (67 after one-hot coding): time of day and calendar, airline, origin, destination and its group, route history, the plane, weather of the scheduled hour, flights scheduled at the airport, how late the flights that left the same airport in the last two hours were, and how late the same plane's previous flight was. Trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December 0.342 (95% interval 0.306-0.376); the dumb answer scores 0.141 and the hour-only rule 0.184. Other models (gradient boosting, random forest, logistic regression) are within the same interval.
8. **Weak spots.** Mornings: recall 8.2% (evenings 53.3%). Newark: recall 24.7% (JFK 45.1%). The cut-off flags a flight mostly because it is late in the day or the airport is already in trouble, so the first flights of the day get few warnings. It warns JetBlue and Southwest flights most (their real late rates are highest) and Alaska, American and Endeavor less than their real risk deserves (Task 6, fairness).
9. **Re-check when.** Weekly recall at the cut-off, over weeks with at least 100 cases, ranged from 17.4% to 53.6% in this quarter. Re-check the model if the weekly recall falls below 17% for two weeks in a row, or if weekly precision falls to the base rate. Tentative: based on 14 weeks.
10. **Human check.** A person confirms the flight status on the airline's feed before messaging a client, handles clients with tight connections first, and never auto-rebooks. Because the score is not a promise (36.0% of flags are real), the message says "possible delay".
11. **Finding.** Delays carry over through the day: how late the flights that just left the same airport were is the strongest signal, ten times stronger than weather. The weather the agency can know two hours ahead gives the same accuracy as the stand-in (change +0.0005, interval -0.0030 to +0.0039).


In [11]:
def card2():
    c = CUT[2]; m = c["test"]; g = GRP[2]; h = HRS[2]; o = ORG[2]; ci = BOOT_AP[2]; mo = t2.groupby(t2.date.dt.month)
    return f'''**CARD 2: cancelled flights**

1. **Prediction.** Will this flight be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all cancelled flights that were flagged), with precision next to it. A cancellation the agency did not see coming leaves the client stranded; an unneeded rebooking offer costs one message. Cancellations are very rare ({pct(c['base'], 2)} of test flights), so accuracy is meaningless and PR-AUC is used.
3. **Cut-off.** Flag the flight when the model score is at least **{c['thr']:.3f}**, chosen on July-September (best F1). On those months it flags {pct(c['val']['flagged_share'])} of flights; on October-December **{pct(m['flagged_share'], 2)}** ({c['per_day']:.1f} flights a day), because the autumn had few cancellations (drift). More capacity: the top 1% of scores ({LAD2.loc[0.01, 'flights/day (top x%)']:.0f} flights a day) gives precision {pct(LAD2.loc[0.01, 'precision (top x%)'])} and recall {pct(LAD2.loc[0.01, 'recall (top x%)'])}.
4. **Action.** Contact the client of each flagged flight and offer a rebooking before the cancellation happens; do not rebook automatically. Rebooking offers are strongest for the short-haul group G1 (precision {g.loc['G1', 'precision %']:.1f}%, recall {g.loc['G1', 'recall %']:.1f}%, {int(g.loc['G1', 'cases'])} cases) and weaker for the long-haul group G2 (precision {g.loc['G2', 'precision %']:.1f}%, recall {g.loc['G2', 'recall %']:.1f}%, {int(g.loc['G2', 'cases'])} cases), where the alert is less reliable. The other groups have too few flights to say anything.
5. **Value (October-December).** {c['flagged']:,} flights flagged; {m['tp']} of them were cancelled (precision {pct(m['precision'])}, 95% interval {pct(c['pci'][0])}-{pct(c['pci'][1])}); they are {pct(m['recall'])} of all {int(t2.y_true.sum())} cancelled flights (recall, interval {pct(c['rci'][0])}-{pct(c['rci'][1])}). Picking the same number of flights at random would catch {pct(c['rand_recall'], 2)} (about {c['rand_tp']:.1f} cases, precision {pct(c['base'], 2)}): the cut-off is **{c['lift']:.0f} times better than random**. The dumb answer catches 0.
6. **Model.** Gradient boosting (best on validation): tree-based ensemble on features (time, airline, airport, weather of the scheduled hour, how late recent flights from the same airport were, the plane), standardised, trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December {ci.loc['best', 'pr_auc']:.3f} (95% interval {ci.loc['best', 'low']:.3f}-{ci.loc['best', 'high']:.3f}); the dumb answer scores {ci.loc['dumb', 'pr_auc']:.4f}. The interval is wide because 121 of the 224 cancellations fall in one week; without that week the score is 0.0071 against 0.0011 for the dumb answer. The three models tried cannot be told apart. Precision stays low in absolute terms: about {c['test']['precision'] * 100:.0f} in 100 flags are real.
8. **Weak spots.** The model found **none** of the {int(o.loc['EWR', 'cases'])} cancellations at Newark (recall {o.loc['EWR', 'recall %']:.0f}%), against {o.loc['JFK', 'recall %']:.0f}% at JFK and {o.loc['LGA', 'recall %']:.0f}% at LaGuardia. By month: October recall {100 * at_threshold(t2[t2.date.dt.month == 10].y_true, t2[t2.date.dt.month == 10].score_best, c['thr'])['recall']:.0f}%, November {100 * at_threshold(t2[t2.date.dt.month == 11].y_true, t2[t2.date.dt.month == 11].score_best, c['thr'])['recall']:.0f}%, December {100 * at_threshold(t2[t2.date.dt.month == 12].y_true, t2[t2.date.dt.month == 12].score_best, c['thr'])['recall']:.0f}%: it finds cancellations when the airport is already in trouble (mostly the single bad day of 18 December) and few in quiet weeks. It flags Republic flights at about four times their real cancellation rate (its cancellations are mostly caught: recall 83.7%).
9. **Re-check when.** Single weeks have too few cancellations to judge (see the weekly table), so pool 4 weeks and judge only a pool with at least 30 cancelled flights. The pooled recall at the cut-off was very uneven in this quarter: {', '.join(f"{i.replace('weeks ', 'weeks ')} {r:.0f}%" for i, r in BLK2.loc[BLK2.cases >= 30, 'recall %'].items())} (blocks with at least 30 cases): close to nothing in quiet weeks, about two thirds in the bad-day block. Re-check the model if, in a bad-day block (at least 30 cancellations and more than 1% of flights flagged), the pooled recall falls below {50 * c['val']['recall']:.0f}%, which is half of the validation recall ({pct(c['val']['recall'])}). Tentative: it rests on one bad day.
10. **Human check.** A person confirms on the airline's feed that the flight is still scheduled before offering a rebooking, never rebooks automatically, handles clients with tight connections first, and treats a flag as "high risk", not "will be cancelled".
11. **Finding.** Cancellations come in bursts, not evenly: {D18['cancelled']} of the {int(t2.y_true.sum())} cancellations of October-December fell on one day, 18 December ({D18['share']*100:.1f}% of that day's flights, mostly Endeavor and Republic at LaGuardia and JFK), and in January-September 27 bad days held 57.5% of all cancellations (Task 2). The weather of that week was ordinary (rain in {W51.rain_hours_pct:.0f}% of hours against a median of {WOTH.rain_hours_pct:.0f}% in the other weeks; gusts above 25 mph in {W51.gust_over_25_pct:.1f}% of hours, the same as weeks 46 to 48, which had 6, 1 and 3 cancellations), so the cause is not in the data. The agency should plan rebooking capacity in bursts around bad days.
'''
display(Markdown(card2()))


**CARD 2: cancelled flights**

1. **Prediction.** Will this flight be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all cancelled flights that were flagged), with precision next to it. A cancellation the agency did not see coming leaves the client stranded; an unneeded rebooking offer costs one message. Cancellations are very rare (0.21% of test flights), so accuracy is meaningless and PR-AUC is used.
3. **Cut-off.** Flag the flight when the model score is at least **0.125**, chosen on July-September (best F1). On those months it flags 5.7% of flights; on October-December **0.77%** (8.8 flights a day), because the autumn had few cancellations (drift). More capacity: the top 1% of scores (11 flights a day) gives precision 6.6% and recall 30.8%.
4. **Action.** Contact the client of each flagged flight and offer a rebooking before the cancellation happens; do not rebook automatically. Rebooking offers are strongest for the short-haul group G1 (precision 7.9%, recall 19.8%, 81 cases) and weaker for the long-haul group G2 (precision 6.0%, recall 25.2%, 143 cases), where the alert is less reliable. The other groups have too few flights to say anything.
5. **Value (October-December).** 809 flights flagged; 52 of them were cancelled (precision 6.4%, 95% interval 2.2%-11.1%); they are 23.2% of all 224 cancelled flights (recall, interval 16.2%-31.3%). Picking the same number of flights at random would catch 0.77% (about 1.7 cases, precision 0.21%): the cut-off is **30 times better than random**. The dumb answer catches 0.
6. **Model.** Gradient boosting (best on validation): tree-based ensemble on features (time, airline, airport, weather of the scheduled hour, how late recent flights from the same airport were, the plane), standardised, trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December 0.049 (95% interval 0.013-0.096); the dumb answer scores 0.0021. The interval is wide because 121 of the 224 cancellations fall in one week; without that week the score is 0.0071 against 0.0011 for the dumb answer. The three models tried cannot be told apart. Precision stays low in absolute terms: about 6 in 100 flags are real.
8. **Weak spots.** The model found **none** of the 39 cancellations at Newark (recall 36%), against 24% at JFK and 19% at LaGuardia. By month: October recall 17%, November 33%, December 24%: it finds cancellations when the airport is already in trouble (mostly the single bad day of 18 December) and few in quiet weeks. It flags Republic flights at about four times their real cancellation rate (its cancellations are mostly caught: recall 83.7%).
9. **Re-check when.** Single weeks have too few cancellations to judge (see the weekly table), so pool 4 weeks and judge only a pool with at least 30 cancelled flights. The pooled recall at the cut-off was very uneven in this quarter: weeks 39-42 23%, weeks 43-46 17%, weeks 51-52 22% (blocks with at least 30 cases): close to nothing in quiet weeks, about two thirds in the bad-day block. Re-check the model if, in a bad-day block (at least 30 cancellations and more than 1% of flights flagged), the pooled recall falls below 25%, which is half of the validation recall (49.8%). Tentative: it rests on one bad day.
10. **Human check.** A person confirms on the airline's feed that the flight is still scheduled before offering a rebooking, never rebooks automatically, handles clients with tight connections first, and treats a flag as "high risk", not "will be cancelled".
11. **Finding.** Cancellations come in bursts, not evenly: 115 of the 224 cancellations of October-December fell on one day, 18 December (9.8% of that day's flights, mostly Endeavor and Republic at LaGuardia and JFK), and in January-September 27 bad days held 57.5% of all cancellations (Task 2). The weather of that week was ordinary (rain in 12% of hours against a median of 11% in the other weeks; gusts above 25 mph in 1.8% of hours, the same as weeks 46 to 48, which had 6, 1 and 3 cancellations), so the cause is not in the data. The agency should plan rebooking capacity in bursts around bad days.


## 8. Both cards together (final cell)

In [12]:
display(Markdown(card1())); display(Markdown("---")); display(Markdown(card2()))

**CARD 1: late or cancelled flights**

1. **Prediction.** Will this flight leave more than 15 minutes late or be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all late-or-cancelled flights that were flagged), with precision next to it (share of flagged flights that really were). A missed late flight leaves a client unwarned; a false warning costs one message and some trust, so recall matters more than precision and a ranking metric (PR-AUC) is used to compare models. Accuracy is not used: "nobody is late" would be 85.9% accurate and useless.
3. **Cut-off.** Flag the flight when the model score is at least **0.336**, chosen on July-September (best F1). On those months it flags 35.3% of flights; on October-December it flags **13.2%** (150 flights a day), because the autumn was calmer (drift). If the agency instead has capacity for a fixed share: the top 20% of scores gives precision 31.3% and recall 44.5% (227 flights a day).
4. **Action.** Message the client of each flagged flight two hours before departure and check their connections. Use the destination groups of Task 3: for the main long-haul group G2 (73,579 test flights, recall 35.2%, precision 35.8%) warn earlier for afternoon and evening flights; for the short-haul group G1 (30,163 flights, recall 25.3%, precision 35.8%) keep rebooking ready, since it has the most cancellations. The group figures are close, so the rule is the same and only the message changes.
5. **Value (October-December).** 13,830 flights flagged; 4,976 of them were real cases (precision 36.0%, 95% interval 33.1%-38.7%); they are 33.9% of all 14,685 late-or-cancelled flights (recall, interval 28.5%-38.8%). Picking the same number of flights at random would catch 13.2% (about 1,943 cases, precision 14.1%): the cut-off is **2.6 times better than random**. The dumb answer catches 0.
6. **Model.** Neural network (two hidden layers of 128 and 64 units), 37 input columns (67 after one-hot coding): time of day and calendar, airline, origin, destination and its group, route history, the plane, weather of the scheduled hour, flights scheduled at the airport, how late the flights that left the same airport in the last two hours were, and how late the same plane's previous flight was. Trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December 0.342 (95% interval 0.306-0.376); the dumb answer scores 0.141 and the hour-only rule 0.184. Other models (gradient boosting, random forest, logistic regression) are within the same interval.
8. **Weak spots.** Mornings: recall 8.2% (evenings 53.3%). Newark: recall 24.7% (JFK 45.1%). The cut-off flags a flight mostly because it is late in the day or the airport is already in trouble, so the first flights of the day get few warnings. It warns JetBlue and Southwest flights most (their real late rates are highest) and Alaska, American and Endeavor less than their real risk deserves (Task 6, fairness).
9. **Re-check when.** Weekly recall at the cut-off, over weeks with at least 100 cases, ranged from 17.4% to 53.6% in this quarter. Re-check the model if the weekly recall falls below 17% for two weeks in a row, or if weekly precision falls to the base rate. Tentative: based on 14 weeks.
10. **Human check.** A person confirms the flight status on the airline's feed before messaging a client, handles clients with tight connections first, and never auto-rebooks. Because the score is not a promise (36.0% of flags are real), the message says "possible delay".
11. **Finding.** Delays carry over through the day: how late the flights that just left the same airport were is the strongest signal, ten times stronger than weather. The weather the agency can know two hours ahead gives the same accuracy as the stand-in (change +0.0005, interval -0.0030 to +0.0039).


---

**CARD 2: cancelled flights**

1. **Prediction.** Will this flight be cancelled? Predicted for every flight two hours before scheduled departure.
2. **Metric and why.** Recall at the cut-off (share of all cancelled flights that were flagged), with precision next to it. A cancellation the agency did not see coming leaves the client stranded; an unneeded rebooking offer costs one message. Cancellations are very rare (0.21% of test flights), so accuracy is meaningless and PR-AUC is used.
3. **Cut-off.** Flag the flight when the model score is at least **0.125**, chosen on July-September (best F1). On those months it flags 5.7% of flights; on October-December **0.77%** (8.8 flights a day), because the autumn had few cancellations (drift). More capacity: the top 1% of scores (11 flights a day) gives precision 6.6% and recall 30.8%.
4. **Action.** Contact the client of each flagged flight and offer a rebooking before the cancellation happens; do not rebook automatically. Rebooking offers are strongest for the short-haul group G1 (precision 7.9%, recall 19.8%, 81 cases) and weaker for the long-haul group G2 (precision 6.0%, recall 25.2%, 143 cases), where the alert is less reliable. The other groups have too few flights to say anything.
5. **Value (October-December).** 809 flights flagged; 52 of them were cancelled (precision 6.4%, 95% interval 2.2%-11.1%); they are 23.2% of all 224 cancelled flights (recall, interval 16.2%-31.3%). Picking the same number of flights at random would catch 0.77% (about 1.7 cases, precision 0.21%): the cut-off is **30 times better than random**. The dumb answer catches 0.
6. **Model.** Gradient boosting (best on validation): tree-based ensemble on features (time, airline, airport, weather of the scheduled hour, how late recent flights from the same airport were, the plane), standardised, trained on flights from January to September 2023.
7. **Score.** PR-AUC on October-December 0.049 (95% interval 0.013-0.096); the dumb answer scores 0.0021. The interval is wide because 121 of the 224 cancellations fall in one week; without that week the score is 0.0071 against 0.0011 for the dumb answer. The three models tried cannot be told apart. Precision stays low in absolute terms: about 6 in 100 flags are real.
8. **Weak spots.** The model found **none** of the 39 cancellations at Newark (recall 36%), against 24% at JFK and 19% at LaGuardia. By month: October recall 17%, November 33%, December 24%: it finds cancellations when the airport is already in trouble (mostly the single bad day of 18 December) and few in quiet weeks. It flags Republic flights at about four times their real cancellation rate (its cancellations are mostly caught: recall 83.7%).
9. **Re-check when.** Single weeks have too few cancellations to judge (see the weekly table), so pool 4 weeks and judge only a pool with at least 30 cancelled flights. The pooled recall at the cut-off was very uneven in this quarter: weeks 39-42 23%, weeks 43-46 17%, weeks 51-52 22% (blocks with at least 30 cases): close to nothing in quiet weeks, about two thirds in the bad-day block. Re-check the model if, in a bad-day block (at least 30 cancellations and more than 1% of flights flagged), the pooled recall falls below 25%, which is half of the validation recall (49.8%). Tentative: it rests on one bad day.
10. **Human check.** A person confirms on the airline's feed that the flight is still scheduled before offering a rebooking, never rebooks automatically, handles clients with tight connections first, and treats a flag as "high risk", not "will be cancelled".
11. **Finding.** Cancellations come in bursts, not evenly: 115 of the 224 cancellations of October-December fell on one day, 18 December (9.8% of that day's flights, mostly Endeavor and Republic at LaGuardia and JFK), and in January-September 27 bad days held 57.5% of all cancellations (Task 2). The weather of that week was ordinary (rain in 12% of hours against a median of 11% in the other weeks; gusts above 25 mph in 1.8% of hours, the same as weeks 46 to 48, which had 6, 1 and 3 cancellations), so the cause is not in the data. The agency should plan rebooking capacity in bursts around bad days.
